# Data Cleaning: DataCo Supply Chain

Reads the raw DataCo order file, checks it, fixes what is wrong, and saves `clean_orders.csv`.
One row is one order line (one product inside an order).

- **Source:** DataCo Smart Supply Chain for Big Data Analysis (Mendeley Data, licence CC BY 4.0)
- **Input:** `DataCoSupplyChainDataset.csv` (raw, 53 columns)
- **Output:** `clean_orders.csv` (29 columns), used by the MySQL load, the SQL analysis and the Power BI dashboard

# Part 0: Setup and Load the Raw File

## 0.1 Connect Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 0.2 Imports and file locations

In [ ]:

import pandas as pd
import os

data_folder = "/content/drive/MyDrive/supply_chain_project/data"
file_path = "/content/drive/MyDrive/data/DataCoSupplyChainDataset.csv"

## 0.3 Load the raw data
The file is not UTF-8 (a normal `read_csv` fails with `UnicodeDecodeError`), so `latin-1` is used.

In [ ]:
# Load the raw file; df stays untouched
df = pd.read_csv(file_path, encoding="latin-1")

# Show all columns when printing
pd.set_option("display.max_columns", None)

# All cleaning is done on df_clean
df_clean = df.copy()

# Part 1: Inspect the Raw Data


## 1.1 Size and column names

In [ ]:
print(df.shape)
print(df.columns.tolist())

(180519, 53)
['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Delivery Status', 'Late_delivery_risk', 'Category Id', 'Category Name', 'Customer City', 'Customer Country', 'Customer Email', 'Customer Fname', 'Customer Id', 'Customer Lname', 'Customer Password', 'Customer Segment', 'Customer State', 'Customer Street', 'Customer Zipcode', 'Department Id', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order City', 'Order Country', 'Order Customer Id', 'order date (DateOrders)', 'Order Id', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Region', 'Order State', 'Order Status', 'Order Zipcode', 'Product Card Id', 'Product Category Id', 'Product Description', 'Product Image', 'Product Name', 'Product Price', 'Product Status', 'shipping d

## 1.2 Data types

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 53 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Type                           180519 non-null  object 
 1   Days for shipping (real)       180519 non-null  int64  
 2   Days for shipment (scheduled)  180519 non-null  int64  
 3   Benefit per order              180519 non-null  float64
 4   Sales per customer             180519 non-null  float64
 5   Delivery Status                180519 non-null  object 
 6   Late_delivery_risk             180519 non-null  int64  
 7   Category Id                    180519 non-null  int64  
 8   Category Name                  180519 non-null  object 
 9   Customer City                  180519 non-null  object 
 10  Customer Country               180519 non-null  object 
 11  Customer Email                 180519 non-null  object 
 12  Customer Fname                

**Result:** both date columns are stored as text (fixed later).

## 1.3 Missing values

In [ ]:
# Null count and null percent (only columns with nulls)
nulls = df.isnull().sum()
print(nulls[nulls > 0])
print()
print((nulls[nulls > 0] / len(df) * 100).round(2))

Customer Lname              8
Customer Zipcode            3
Order Zipcode          155679
Product Description    180519
dtype: int64

Customer Lname           0.00
Customer Zipcode         0.00
Order Zipcode           86.24
Product Description    100.00
dtype: float64


**Result:** four columns have nulls: `Product Description` (100%), `Order Zipcode` (86%), `Customer Lname` (8 rows) and `Customer Zipcode` (3 rows).

## 1.4 Duplicate rows

In [ ]:
# Count of fully duplicated rows
print(df.duplicated().sum())

0


## 1.5 Grain check
Compares the number of rows, different orders and different order lines.

In [ ]:
# Rows vs different orders vs different order lines
print(len(df))
print(df["Order Id"].nunique())
print(df["Order Item Id"].nunique())

180519
65752
180519


## 1.6 Date format
Tests the order date in both formats: month first (`%m/%d/%Y`) and day first (`%d/%m/%Y`).

In [ ]:
# Test which date order is correct (nothing is changed in df)
print(df["order date (DateOrders)"].dtype)
print(df["order date (DateOrders)"].head(3))

month_first = pd.to_datetime(df["order date (DateOrders)"], format="%m/%d/%Y %H:%M", errors="coerce")
day_first = pd.to_datetime(df["order date (DateOrders)"], format="%d/%m/%Y %H:%M", errors="coerce")
print("failed with month first:", month_first.isnull().sum())
print("failed with day first:", day_first.isnull().sum())

object
0    1/31/2018 22:56
1    1/13/2018 12:27
2    1/13/2018 12:06
Name: order date (DateOrders), dtype: object
failed with month first: 0
failed with day first: 109416


**Result:** month first gives 0 failures, day first gives 109,416. Dates are **month first**.

## 1.7 Date range and shipping date
Shows the earliest and latest order date and tests the shipping date format.

In [ ]:
# Earliest and latest order date; test the shipping date format
print(month_first.min())
print(month_first.max())

ship_month_first = pd.to_datetime(df["shipping date (DateOrders)"], format="%m/%d/%Y %H:%M", errors="coerce")
print("shipping date failed with month first:", ship_month_first.isnull().sum())

2015-01-01 00:00:00
2018-01-31 23:38:00
shipping date failed with month first: 0


**Result:** order dates run from 2015-01-01 to 2018-01-31. The shipping date parses with 0 failures.

## 1.8 Category columns

In [ ]:
# Values and counts of key category columns
print(df["Shipping Mode"].value_counts(dropna=False))
print()
print(df["Delivery Status"].value_counts(dropna=False))
print()
print(df["Order Status"].value_counts(dropna=False))

Shipping Mode
Standard Class    107752
Second Class       35216
First Class        27814
Same Day            9737
Name: count, dtype: int64

Delivery Status
Late delivery        98977
Advance shipping     41592
Shipping on time     32196
Shipping canceled     7754
Name: count, dtype: int64

Order Status
COMPLETE           59491
PENDING_PAYMENT    39832
PROCESSING         21902
PENDING            20227
CLOSED             19616
ON_HOLD             9804
SUSPECTED_FRAUD     4062
CANCELED            3692
PAYMENT_REVIEW      1893
Name: count, dtype: int64


**Result:** four shipping modes and nine order statuses. No typo categories.

## 1.9 Hidden spaces, fake empty values and broken letters

In [ ]:
print((df["Shipping Mode"] != df["Shipping Mode"].str.strip()).sum())
print(df["Customer Email"].value_counts().head(3))
print(df["Customer City"].str.contains("Ã").sum())

0
Customer Email
XXXXXXXXX    180519
Name: count, dtype: int64
0


**Result:** no hidden spaces in `Shipping Mode`. `Customer Email` is `XXXXXXXXX` in every row (no information, dropped later). No broken letters (`Ã`) in city names.

# Part 2: Clean the Column Names
**Goal:** `snake_case` names (lowercase, underscores, no brackets) so SQL can use them without backticks.

## 2.1 Fix: standardise all names
Example: `Days for shipping (real)` becomes `days_for_shipping_real`.

In [ ]:
# Strip spaces, lowercase, replace spaces with underscores, remove brackets
df_clean.columns = df_clean.columns.str.strip()
df_clean.columns = df_clean.columns.str.lower()
df_clean.columns = df_clean.columns.str.replace(" ", "_")
df_clean.columns = df_clean.columns.str.replace("(", "", regex=False)
df_clean.columns = df_clean.columns.str.replace(")", "", regex=False)

## 2.2 Fix: rename four unclear columns

In [ ]:
# Shorter, clearer names for date and shipping-day columns
df_clean = df_clean.rename(columns={
    "order_date_dateorders": "order_date",
    "shipping_date_dateorders": "shipping_date",
    "days_for_shipping_real": "shipping_days_real",
    "days_for_shipment_scheduled": "shipping_days_scheduled",
})

## 2.3 Check after

In [ ]:
# Verify the new column names
print(df_clean.columns[df_clean.columns.str.contains(" ")])
print(df_clean.columns[df_clean.columns.str.contains("(", regex=False)])
print("order_date" in df_clean.columns)
print("shipping_days_real" in df_clean.columns)
print(df_clean.columns.duplicated().sum())
print(df_clean.shape)

Index([], dtype='object')
Index([], dtype='object')
True
True
0
(180519, 53)


# Part 3: Convert the Date Columns
**Goal:** turn `order_date` and `shipping_date` from text into real dates, so delay days and monthly trends can be calculated.  
The month-first format and 0 failures were already confirmed in Part 1 (1.6 and 1.7).

## 3.1 Fix: convert to datetime
`errors="coerce"` is not used, so any unexpected value raises an error instead of being hidden.

In [ ]:
# Convert text to datetime (month first)
df_clean["order_date"] = pd.to_datetime(df_clean["order_date"], format="%m/%d/%Y %H:%M")
df_clean["shipping_date"] = pd.to_datetime(df_clean["shipping_date"], format="%m/%d/%Y %H:%M")

## 3.2 Check after

In [ ]:
# Verify data type, empty dates, date range and orders per year
print(df_clean["order_date"].dtype)
print(df_clean["shipping_date"].dtype)
print(df_clean["order_date"].isnull().sum())
print(df_clean["shipping_date"].isnull().sum())
print(df_clean["order_date"].min())
print(df_clean["order_date"].max())
print(df_clean["order_date"].dt.year.value_counts().sort_index())

datetime64[ns]
datetime64[ns]
0
0
2015-01-01 00:00:00
2018-01-31 23:38:00
order_date
2015    62650
2016    62550
2017    53196
2018     2123
Name: count, dtype: int64


# Part 4: Standardise the Text Columns


## 4.1 Check first: hidden spaces
A value that changes after `strip()` has leading or trailing spaces.

In [ ]:
# Count values with leading/trailing spaces in each text column
print((df_clean["shipping_mode"] != df_clean["shipping_mode"].str.strip()).sum())
print((df_clean["delivery_status"] != df_clean["delivery_status"].str.strip()).sum())
print((df_clean["category_name"] != df_clean["category_name"].str.strip()).sum())
print((df_clean["customer_segment"] != df_clean["customer_segment"].str.strip()).sum())
print((df_clean["department_name"] != df_clean["department_name"].str.strip()).sum())
print((df_clean["market"] != df_clean["market"].str.strip()).sum())
print((df_clean["order_country"] != df_clean["order_country"].str.strip()).sum())
print((df_clean["order_region"] != df_clean["order_region"].str.strip()).sum())
print((df_clean["order_status"] != df_clean["order_status"].str.strip()).sum())
print((df_clean["product_name"] != df_clean["product_name"].str.strip()).sum())

0
0
1475
0
362
0
0
17925
0
1774


**Result:** found in `category_name` (1,475), `department_name` (362), `order_region` (17,925) and `product_name` (1,774).

## 4.2 Check first: spelling and case variants
Compares the number of different values before and after `strip().lower()`.

In [ ]:
# Different values: original vs stripped and lowercased
print(df_clean["shipping_mode"].nunique(), df_clean["shipping_mode"].str.strip().str.lower().nunique())
print(df_clean["delivery_status"].nunique(), df_clean["delivery_status"].str.strip().str.lower().nunique())
print(df_clean["customer_segment"].nunique(), df_clean["customer_segment"].str.strip().str.lower().nunique())
print(df_clean["market"].nunique(), df_clean["market"].str.strip().str.lower().nunique())
print(df_clean["order_status"].nunique(), df_clean["order_status"].str.strip().str.lower().nunique())
print(df_clean["order_region"].nunique(), df_clean["order_region"].str.strip().str.lower().nunique())

4 4
4 4
3 3
5 5
9 9
23 23


**Result:** the counts are the same, so there are no case variants.

## 4.3 Check first: broken letters

In [ ]:
# Count values with the broken-letter symbol
print(df_clean["order_country"].str.contains("Ã").sum())
print(df_clean["product_name"].str.contains("Ã").sum())

0
0


## 4.4 Fix: remove hidden spaces
Case is not changed, because no case variants were found.

In [ ]:
# Remove leading/trailing spaces from the ten text columns
df_clean["shipping_mode"] = df_clean["shipping_mode"].str.strip()
df_clean["delivery_status"] = df_clean["delivery_status"].str.strip()
df_clean["category_name"] = df_clean["category_name"].str.strip()
df_clean["customer_segment"] = df_clean["customer_segment"].str.strip()
df_clean["department_name"] = df_clean["department_name"].str.strip()
df_clean["market"] = df_clean["market"].str.strip()
df_clean["order_country"] = df_clean["order_country"].str.strip()
df_clean["order_region"] = df_clean["order_region"].str.strip()
df_clean["order_status"] = df_clean["order_status"].str.strip()
df_clean["product_name"] = df_clean["product_name"].str.strip()

## 4.5 Check after

In [ ]:
# Verify shipping_mode and shape
print((df_clean["shipping_mode"] != df_clean["shipping_mode"].str.strip()).sum())
print(df_clean["shipping_mode"].nunique())
print(df_clean.shape)

0
4
(180519, 53)


# Part 5: Drop Unnecessary Columns
**Goal:** keep the 26 columns needed for analysis and drop the other 27.

**Dropped groups:** personal data (names, street, email, password), duplicate columns, empty or constant columns, and columns that are too detailed.

## 5.1 Check first: are the duplicate columns exact copies?
100 means the two columns are identical.

In [ ]:
# Percent of rows where the two columns are identical
print((df_clean["benefit_per_order"] == df_clean["order_profit_per_order"]).mean() * 100)
print((df_clean["sales_per_customer"] == df_clean["order_item_total"]).mean() * 100)
print((df_clean["customer_id"] == df_clean["order_customer_id"]).mean() * 100)
print((df_clean["category_id"] == df_clean["product_category_id"]).mean() * 100)
print((df_clean["order_item_cardprod_id"] == df_clean["product_card_id"]).mean() * 100)
print((df_clean["order_item_product_price"] == df_clean["product_price"]).mean() * 100)

100.0
100.0
100.0
100.0
100.0
100.0


**Result:** all six pairs match 100%, so the first column of each pair can be dropped.

## 5.2 Check first: empty or constant columns

In [ ]:
# Null percent and number of different values
print(df_clean["product_description"].isnull().mean() * 100)
print(df_clean["order_zipcode"].isnull().mean() * 100)
print(df_clean["product_status"].nunique())

100.0
86.23967560201419
1


**Result:** `product_description` is 100% empty, `order_zipcode` is 86% empty, and `product_status` has a single value.

## 5.3 Check first: the 26 columns to keep
A keep-list is safer than a drop-list. Selecting the columns also proves there is no typo in any name.

In [ ]:
# Columns to keep
keep_columns = [
    "shipping_days_real", "shipping_days_scheduled", "delivery_status",
    "late_delivery_risk", "category_id", "category_name", "customer_id",
    "customer_segment", "department_name", "market", "order_country",
    "order_date", "order_id", "order_item_discount", "order_item_id",
    "order_item_quantity", "sales", "order_item_total",
    "order_profit_per_order", "order_region", "order_status",
    "product_card_id", "product_name", "product_price",
    "shipping_date", "shipping_mode",
]
print(len(keep_columns))
print(df_clean[keep_columns].shape)

26
(180519, 26)


**Result:** 26 columns selected, shape (180,519, 26).

## 5.4 Fix: keep only the selected columns
The raw `df` still has all 53 columns.

In [ ]:
# Keep 26 columns
df_clean = df_clean[keep_columns].copy()

## 5.5 Check after

In [ ]:
# Verify shape, column list and that the email column is removed
print(df_clean.shape)
print(df_clean.columns.tolist())
print("customer_email" in df_clean.columns)

(180519, 26)
['shipping_days_real', 'shipping_days_scheduled', 'delivery_status', 'late_delivery_risk', 'category_id', 'category_name', 'customer_id', 'customer_segment', 'department_name', 'market', 'order_country', 'order_date', 'order_id', 'order_item_discount', 'order_item_id', 'order_item_quantity', 'sales', 'order_item_total', 'order_profit_per_order', 'order_region', 'order_status', 'product_card_id', 'product_name', 'product_price', 'shipping_date', 'shipping_mode']
False


# Part 6: Nulls and Duplicates


## 6.1 Check: nulls

In [ ]:
# Total nulls and columns with nulls
print(df_clean.isnull().sum().sum())

nulls_clean = df_clean.isnull().sum()
print(nulls_clean[nulls_clean > 0])

0
Series([], dtype: int64)


## 6.2 Check: duplicates
Checks duplicate rows and repeated `order_item_id`.

In [ ]:
# Duplicate rows and duplicate order lines
print(df_clean.duplicated().sum())
print(df_clean["order_item_id"].duplicated().sum())

0
0


# Part 7: Impossible Values
**Goal:** find values that cannot be true (for example a negative quantity). Unusual is not the same as wrong: a negative profit is a real loss and is kept.

## 7.1 Value ranges

In [ ]:
# Summary statistics of numeric columns
print(df_clean[["shipping_days_real", "shipping_days_scheduled", "order_item_quantity"]].describe())
print()
print(df_clean[["sales", "product_price", "order_item_total", "order_item_discount"]].describe())

       shipping_days_real  shipping_days_scheduled  order_item_quantity
count       180519.000000            180519.000000        180519.000000
mean             3.497654                 2.931847             2.127638
std              1.623722                 1.374449             1.453451
min              0.000000                 0.000000             1.000000
25%              2.000000                 2.000000             1.000000
50%              3.000000                 4.000000             1.000000
75%              5.000000                 4.000000             3.000000
max              6.000000                 4.000000             5.000000

               sales  product_price  order_item_total  order_item_discount
count  180519.000000  180519.000000     180519.000000        180519.000000
mean      203.772096     141.232550        183.107609            20.664741
std       132.273077     139.732492        120.043670            21.800901
min         9.990000       9.990000          7.4900

## 7.2 Count impossible values
A shipping time of 0 days is valid, because Same Day shipping exists.

In [ ]:
# Count negative or zero values that cannot be true
print((df_clean["order_item_quantity"] <= 0).sum())
print((df_clean["product_price"] <= 0).sum())
print((df_clean["sales"] <= 0).sum())
print((df_clean["order_item_discount"] < 0).sum())
print((df_clean["shipping_days_real"] < 0).sum())
print((df_clean["shipping_days_scheduled"] < 0).sum())

0
0
0
0
0
0


## 7.3 Shipped before ordered?

In [ ]:
# Rows where shipping date is earlier than order date
print((df_clean["shipping_date"] < df_clean["order_date"]).sum())

0


## 7.4 Negative profit
Unusual is not the same as wrong, so these rows are only counted, not deleted.

In [ ]:
# Count lines with negative profit
print((df_clean["order_profit_per_order"] < 0).sum())

33784


**Result:** 33,784 lines have a negative profit. These are real losses (discounts or selling below cost) and are kept. No rows were removed in this part. Canceled orders are flagged, not deleted.

# Part 8: Create Business Columns
**Goal:** add `delay_days`, `is_late` and `is_canceled`, calculated once so every later step uses the same definition.

**Rules:** `delay_days` = real days - scheduled days (negative = early). A line is late only if `delay_days > 0`; exactly 0 is on time.

## 8.1 Create `delay_days` and `is_late`
`is_late` is 1 or 0, so its average is the late percent.

In [ ]:
# Delay in days and late flag (1 = late, 0 = not late)
df_clean["delay_days"] = df_clean["shipping_days_real"] - df_clean["shipping_days_scheduled"]
df_clean["is_late"] = (df_clean["delay_days"] > 0).astype(int)

## 8.2 Check after

In [ ]:
# Delay range, late counts and late percent
print(df_clean["delay_days"].min())
print(df_clean["delay_days"].max())
print(df_clean["is_late"].value_counts())
print(df_clean["is_late"].mean() * 100)

-2
4
is_late
1    103400
0     77119
Name: count, dtype: int64
57.279289160697765


**Result:** `delay_days` ranges from -2 to 4. 103,400 lines are late (57.28%). This is by order line, not by order.

## 8.3 Compare with the dataset's own labels
Compares `is_late` with `late_delivery_risk` and `delivery_status`.

In [ ]:
# Validate is_late against the dataset's own labels
print((df_clean["is_late"] == df_clean["late_delivery_risk"]).mean() * 100)
print(df_clean.groupby("delivery_status")["is_late"].mean())

97.54984239886106
delivery_status
Advance shipping     0.000000
Late delivery        1.000000
Shipping canceled    0.570415
Shipping on time     0.000000
Name: is_late, dtype: float64


**Result:** `is_late` matches `late_delivery_risk` in 97.55% of rows. `Late delivery` is always late, `Advance shipping` and `Shipping on time` are never late. The difference comes from `Shipping canceled` lines (57% flagged late by our rule).

## 8.4 Flag canceled orders
`is_canceled` = 1 where `order_status` is `CANCELED`. Rows are flagged, not deleted, so SQL can exclude them with `WHERE is_canceled = 0`.

In [ ]:
# 1 = canceled order line, 0 = all others
df_clean["is_canceled"] = (df_clean["order_status"] == "CANCELED").astype(int)
print(df_clean["is_canceled"].sum())
print(df_clean["is_canceled"].mean() * 100)

3692
2.0452140771885507


**Result:** 3,692 canceled lines (2.05%).

The `Shipping canceled` label covers 7,754 lines, which equals these 3,692 canceled orders plus 4,062 `SUSPECTED_FRAUD` orders. The suspected-fraud lines are not flagged here, so they stay in the delivery KPIs. Leaving them out moves the late rate from 57.28% to 57.29%.

## 8.5 Final look at the new columns

In [ ]:
# Check the new columns row by row
print(df_clean.shape)
df_clean[["shipping_days_real", "shipping_days_scheduled", "delay_days", "is_late", "is_canceled"]].head()

(180519, 29)


,shipping_days_real,shipping_days_scheduled,delay_days,is_late,is_canceled
0,3,4,-1,0,0
1,5,4,1,1,0
2,4,4,0,0,0
3,3,4,-1,0,0
4,2,4,-2,0,0


**Result:** shape is now 180,519 x 29 (26 kept + 3 new).

# Part 9: Checks Before Loading into MySQL
**Goal:** confirm each id has one set of details (for primary keys), text lengths, and money decimals, so the MySQL data types are correct.

## 9.1 One segment per customer

In [ ]:
# Max number of segments per customer
print(df_clean.groupby("customer_id")["customer_segment"].nunique().max())

1


## 9.2 One set of details per product

In [ ]:
# Max number of different values per product
print(df_clean.groupby("product_card_id")["product_name"].nunique().max())
print(df_clean.groupby("product_card_id")["category_id"].nunique().max())
print(df_clean.groupby("product_card_id")["category_name"].nunique().max())
print(df_clean.groupby("product_card_id")["department_name"].nunique().max())
print(df_clean.groupby("product_card_id")["product_price"].nunique().max())

1
1
1
1
1


**Result:** `1` for all five columns, so a `products` table is possible.

## 9.3 Counts to match in MySQL

In [ ]:
# Rows, orders, customers, products
print(len(df_clean))
print(df_clean["order_id"].nunique())
print(df_clean["customer_id"].nunique())
print(df_clean["product_card_id"].nunique())

180519
65752
20652
118


## 9.4 Longest text values

In [ ]:
# Longest text value in each column
print(df_clean["product_name"].str.len().max())
print(df_clean["category_name"].str.len().max())
print(df_clean["department_name"].str.len().max())
print(df_clean["order_country"].str.len().max())
print(df_clean["order_region"].str.len().max())
print(df_clean["delivery_status"].str.len().max())
print(df_clean["order_status"].str.len().max())
print(df_clean["shipping_mode"].str.len().max())
print(df_clean["customer_segment"].str.len().max())
print(df_clean["market"].str.len().max())

45
20
17
31
15
17
15
14
11
12


**Result:** all values fit the planned MySQL sizes (for example `product_name` max 45 vs size 100).

## 9.5 Check: decimals in money columns
Compares each value with the same value rounded to 2 decimals.

In [ ]:
# Largest difference between value and value rounded to 2 decimals
print((df_clean["product_price"] - df_clean["product_price"].round(2)).abs().max())
print((df_clean["sales"] - df_clean["sales"].round(2)).abs().max())
print((df_clean["order_item_total"] - df_clean["order_item_total"].round(2)).abs().max())
print((df_clean["order_item_discount"] - df_clean["order_item_discount"].round(2)).abs().max())
print((df_clean["order_profit_per_order"] - df_clean["order_profit_per_order"].round(2)).abs().max())

1.70999999227206e-05
1.70999999227206e-05
2.4400000029345392e-05
6.100000007336348e-06
5.899999996472616e-05


**Result:** money columns have tiny extra decimals (up to about 0.00006). They are rounded in 9.6 to match `DECIMAL(10,2)` in MySQL.

## 9.6 Fix: round money columns to 2 decimals

In [ ]:
# Round money columns to 2 decimals
df_clean["product_price"] = df_clean["product_price"].round(2)
df_clean["sales"] = df_clean["sales"].round(2)
df_clean["order_item_total"] = df_clean["order_item_total"].round(2)
df_clean["order_item_discount"] = df_clean["order_item_discount"].round(2)
df_clean["order_profit_per_order"] = df_clean["order_profit_per_order"].round(2)

## 9.7 Check after

In [ ]:
# Repeat the decimal check
print((df_clean["product_price"] - df_clean["product_price"].round(2)).abs().max())
print((df_clean["sales"] - df_clean["sales"].round(2)).abs().max())
print((df_clean["order_item_total"] - df_clean["order_item_total"].round(2)).abs().max())
print((df_clean["order_item_discount"] - df_clean["order_item_discount"].round(2)).abs().max())
print((df_clean["order_profit_per_order"] - df_clean["order_profit_per_order"].round(2)).abs().max())

0.0
0.0
0.0
0.0
0.0


# Part 10: Final Checks and Save
**Goal:** confirm the table is ready and save `clean_orders.csv`. MySQL and Power BI read only this file.

## 10.1 Final checks

In [ ]:
# Final validation: shape, nulls, duplicates, data types, columns
print(df_clean.shape)
print(df_clean.isnull().sum().sum())
print(df_clean.duplicated().sum())
print(df_clean["order_item_id"].duplicated().sum())
print()
print(df_clean.dtypes)
print()
print(df_clean.columns.tolist())

(180519, 29)
0
0
0

shipping_days_real                  int64
shipping_days_scheduled             int64
delivery_status                    object
late_delivery_risk                  int64
category_id                         int64
category_name                      object
customer_id                         int64
customer_segment                   object
department_name                    object
market                             object
order_country                      object
order_date                 datetime64[ns]
order_id                            int64
order_item_discount               float64
order_item_id                       int64
order_item_quantity                 int64
sales                             float64
order_item_total                  float64
order_profit_per_order            float64
order_region                       object
order_status                       object
product_card_id                     int64
product_name                       object
product_price 

## 10.2 Summary numbers for the cleaning log

In [ ]:
# Key numbers for the cleaning log
print("raw rows and columns:", df.shape)
print("clean rows and columns:", df_clean.shape)
print("late percent (by order line):", round(df_clean["is_late"].mean() * 100, 2))
print("canceled order lines:", df_clean["is_canceled"].sum())
print("different orders:", df_clean["order_id"].nunique())
print("total sales:", round(df_clean["sales"].sum(), 2))

raw rows and columns: (180519, 53)
clean rows and columns: (180519, 29)
late percent (by order line): 57.28
canceled order lines: 3692
different orders: 65752
total sales: 36784734.31


**Result:** raw 180,519 x 53, clean 180,519 x 29, late 57.28%, 3,692 canceled lines, 65,752 orders, total sales 36,784,734.31.

## 10.3 Save `clean_orders.csv`
`index=False` avoids an extra row-number column. Dates are saved as `YYYY-MM-DD HH:MM:SS`, which MySQL reads directly.

In [ ]:
# Save the clean file
os.makedirs(data_folder, exist_ok=True)
df_clean.to_csv(data_folder + "/clean_orders.csv", index=False)

## 10.4 Check the saved file
Reads the file back and compares its shape and size.

In [ ]:
# Read the saved file back and check shape and size
check = pd.read_csv(data_folder + "/clean_orders.csv")
print(check.shape)
print(check.columns.tolist())

size_mb = os.path.getsize(data_folder + "/clean_orders.csv") / (1024 * 1024)
print(round(size_mb, 1), "MB")

(180519, 29)
['shipping_days_real', 'shipping_days_scheduled', 'delivery_status', 'late_delivery_risk', 'category_id', 'category_name', 'customer_id', 'customer_segment', 'department_name', 'market', 'order_country', 'order_date', 'order_id', 'order_item_discount', 'order_item_id', 'order_item_quantity', 'sales', 'order_item_total', 'order_profit_per_order', 'order_region', 'order_status', 'product_card_id', 'product_name', 'product_price', 'shipping_date', 'shipping_mode', 'delay_days', 'is_late', 'is_canceled']
42.7 MB
